In [2]:
# ==========================================================
# ARBRE DE DÉCISION FROM SCRATCH EN PYTHON
# ==========================================================
#
# Un arbre de décision est un algorithme
# de Machine Learning supervisé.
#
# ----------------------------------------------------------
# IL EST UTILISÉ POUR :
# ----------------------------------------------------------
#
# ✔ Classification
# ✔ Régression
#
# Exemple classification :
#
# -> Spam / Non Spam
# -> Oui / Non
# -> Malade / Sain
#
# Exemple régression :
#
# -> prédire un prix
# -> prédire une température
#
# ----------------------------------------------------------
# DANS CE CODE :
# ----------------------------------------------------------
#
# Nous allons construire un arbre
# de décision pour classification binaire.
#
# ----------------------------------------------------------
# OBJECTIF :
# ----------------------------------------------------------
#
# prédire :
#
# Jouer = Oui / Non
#
# selon :
#
# - Meteo
# - Temperature
# - Humidite
# - Vent
#
# ----------------------------------------------------------
# CET ALGORITHME UTILISE :
# ----------------------------------------------------------
#
# ✔ Entropie
# ✔ Information Gain
# ✔ Construction récursive
#

#
# ==========================================================

# ==========================================================
# IMPORTATION DES LIBRARIES
# ==========================================================

# pandas :
# utilisé pour manipuler les tableaux
# de données
import pandas as pd

# numpy :
# utilisé pour les calculs mathématiques
import numpy as np

# Counter :
# utilisé pour compter les occurrences
# des classes
from collections import Counter

# ==========================================================
# 1. DATASET
# ==========================================================

# ----------------------------------------------------------
# Exemple :
# ----------------------------------------------------------
#
# Décider si on joue au tennis.
#
# ----------------------------------------------------------
# FEATURES :
# ----------------------------------------------------------
#
# Meteo
# Temperature
# Humidite
# Vent
#
# ----------------------------------------------------------
# VARIABLE CIBLE :
# ----------------------------------------------------------
#
# Jouer
#
# Oui  -> jouer
# Non  -> ne pas jouer
#
# ----------------------------------------------------------

data = {

    # ------------------------------------------------------
    # FEATURE : MÉTÉO
    # ------------------------------------------------------
    #
    # Valeurs possibles :
    #
    # Soleil
    # Nuageux
    # Pluie
    #
    # ------------------------------------------------------

    'Meteo': [

        'Soleil',
        'Soleil',
        'Nuageux',
        'Pluie',
        'Pluie',
        'Pluie',
        'Nuageux',
        'Soleil',
        'Soleil',
        'Pluie'
    ],

    # ------------------------------------------------------
    # FEATURE : TEMPÉRATURE
    # ------------------------------------------------------
    #
    # Valeurs :
    #
    # Chaud
    # Moyen
    # Froid
    #
    # ------------------------------------------------------

    'Temperature': [

        'Chaud',
        'Chaud',
        'Chaud',
        'Moyen',
        'Froid',
        'Froid',
        'Froid',
        'Moyen',
        'Froid',
        'Moyen'
    ],

    # ------------------------------------------------------
    # FEATURE : HUMIDITÉ
    # ------------------------------------------------------

    'Humidite': [

        'Haute',
        'Haute',
        'Haute',
        'Haute',
        'Normale',
        'Normale',
        'Normale',
        'Haute',
        'Normale',
        'Normale'
    ],

    # ------------------------------------------------------
    # FEATURE : VENT
    # ------------------------------------------------------

    'Vent': [

        'Faible',
        'Fort',
        'Faible',
        'Faible',
        'Faible',
        'Fort',
        'Fort',
        'Faible',
        'Faible',
        'Faible'
    ],

    # ------------------------------------------------------
    # VARIABLE CIBLE
    # ------------------------------------------------------
    #
    # Oui = jouer
    # Non = ne pas jouer
    #
    # ------------------------------------------------------

    'Jouer': [

        'Non',
        'Non',
        'Oui',
        'Oui',
        'Oui',
        'Non',
        'Oui',
        'Non',
        'Oui',
        'Oui'
    ]
}

# ----------------------------------------------------------
# Transformer dictionnaire -> DataFrame
# ----------------------------------------------------------

df = pd.DataFrame(data)

# ----------------------------------------------------------
# Afficher dataset
# ----------------------------------------------------------

print("===== DATASET =====")

print(df)

# ==========================================================
# 2. ENTROPIE
# ==========================================================

def entropy(y):

    """
    ------------------------------------------------------
    ENTROPIE
    ------------------------------------------------------

    L'entropie mesure le désordre
    dans les données.

    ------------------------------------------------------
    CAS 1 :
    ------------------------------------------------------

    Oui Oui Oui Oui

    -> entropie faible
    -> données pures

    ------------------------------------------------------
    CAS 2 :
    ------------------------------------------------------

    Oui Non Oui Non

    -> entropie élevée
    -> données mélangées

    ------------------------------------------------------
    OBJECTIF :
    ------------------------------------------------------

    construire des groupes
    les plus purs possibles.

    ------------------------------------------------------
    FORMULE :
    ------------------------------------------------------

    Entropy(S) = - Σ p log2(p)

    ------------------------------------------------------
    p :
    ------------------------------------------------------

    probabilité d'une classe

    Exemple :

    Oui = 6/10
    Non = 4/10
    """

    # ------------------------------------------------------
    # Compter les classes
    # ------------------------------------------------------
    #
    # Exemple :
    #
    # Oui = 6
    # Non = 4
    #
    # ------------------------------------------------------

    counts = Counter(y)

    # ------------------------------------------------------
    # Nombre total d'exemples
    # ------------------------------------------------------

    total = len(y)

    # ------------------------------------------------------
    # Initialisation entropie
    # ------------------------------------------------------

    ent = 0

    # ------------------------------------------------------
    # Parcourir chaque classe
    # ------------------------------------------------------

    for count in counts.values():

        # --------------------------------------------------
        # Calcul probabilité
        # --------------------------------------------------

        p = count / total

        # --------------------------------------------------
        # Formule entropie
        # --------------------------------------------------

        ent -= p * np.log2(p)

    # ------------------------------------------------------
    # Retourner entropie finale
    # ------------------------------------------------------

    return ent

# ==========================================================
# 3. GAIN D'INFORMATION
# ==========================================================

def information_gain(data, feature, target):

    """
    ------------------------------------------------------
    INFORMATION GAIN
    ------------------------------------------------------

    Le gain d'information mesure :

    combien une feature réduit
    le désordre.

    ------------------------------------------------------
    PLUS LE GAIN EST GRAND :
    ------------------------------------------------------

    -> meilleure séparation

    -> meilleure feature

    ------------------------------------------------------
    FORMULE :
    ------------------------------------------------------

    Gain =
    Entropie totale
    -
    Entropie pondérée
    """

    # ------------------------------------------------------
    # Calcul entropie totale
    # ------------------------------------------------------

    total_entropy = entropy(data[target])

    # ------------------------------------------------------
    # Récupérer valeurs uniques
    # ------------------------------------------------------

    values = data[feature].unique()

    # ------------------------------------------------------
    # Variable entropie pondérée
    # ------------------------------------------------------

    weighted_entropy = 0

    # ------------------------------------------------------
    # Parcourir chaque valeur
    # ------------------------------------------------------

    for value in values:

        # --------------------------------------------------
        # Créer sous-ensemble
        # --------------------------------------------------
        #
        # Exemple :
        #
        # Meteo = Soleil
        #
        # --------------------------------------------------

        subset = data[data[feature] == value]

        # --------------------------------------------------
        # Entropie du sous-ensemble
        # --------------------------------------------------

        subset_entropy = entropy(subset[target])

        # --------------------------------------------------
        # Calcul poids
        # --------------------------------------------------
        #
        # poids =
        #
        # taille sous-ensemble
        # -------------------
        # taille totale
        #
        # --------------------------------------------------

        weight = len(subset) / len(data)

        # --------------------------------------------------
        # Ajouter entropie pondérée
        # --------------------------------------------------

        weighted_entropy += weight * subset_entropy

    # ------------------------------------------------------
    # Calcul gain final
    # ------------------------------------------------------

    gain = total_entropy - weighted_entropy

    return gain

# ==========================================================
# 4. CHOISIR MEILLEURE FEATURE
# ==========================================================

def best_feature(data, features, target):

    """
    ------------------------------------------------------
    Cette fonction teste
    toutes les features.
    ------------------------------------------------------

    Puis elle choisit :

    -> la feature avec le plus grand gain
    """

    # ------------------------------------------------------
    # Dictionnaire des gains
    # ------------------------------------------------------

    gains = {}

    print("\n===== GAINS =====")

    # ------------------------------------------------------
    # Tester chaque feature
    # ------------------------------------------------------

    for feature in features:

        # --------------------------------------------------
        # Calcul gain
        # --------------------------------------------------

        gain = information_gain(
            data,
            feature,
            target
        )

        # --------------------------------------------------
        # Stocker gain
        # --------------------------------------------------

        gains[feature] = gain

        # --------------------------------------------------
        # Affichage gain
        # --------------------------------------------------

        print(f"{feature} : {gain:.4f}")

    # ------------------------------------------------------
    # Choisir gain maximal
    # ------------------------------------------------------

    return max(gains, key=gains.get)

# ==========================================================
# 5. CONSTRUCTION ARBRE
# ==========================================================

def decision_tree(data, features, target):

    """
    ------------------------------------------------------
    Fonction principale
    ------------------------------------------------------

    Construction récursive
    de l'arbre de décision.
    """

    # ------------------------------------------------------
    # Classes présentes
    # ------------------------------------------------------

    labels = data[target]

    # ======================================================
    # CAS D'ARRÊT 1
    # ======================================================
    #
    # Toutes les classes identiques
    #
    # Exemple :
    #
    # Oui Oui Oui
    #
    # -> feuille finale
    #
    # ======================================================

    if len(np.unique(labels)) == 1:

        return labels.iloc[0]

    # ======================================================
    # CAS D'ARRÊT 2
    # ======================================================
    #
    # Plus de features disponibles
    #
    # ======================================================

    if len(features) == 0:

        # retourner classe majoritaire

        return labels.mode()[0]

    # ======================================================
    # Recherche meilleure feature
    # ======================================================

    best = best_feature(
        data,
        features,
        target
    )

    # ======================================================
    # Création arbre
    # ======================================================

    tree = {best: {}}

    # ======================================================
    # Valeurs possibles
    # ======================================================

    values = data[best].unique()

    # ======================================================
    # Construire sous-arbres
    # ======================================================

    for value in values:

        # --------------------------------------------------
        # Créer sous-ensemble
        # --------------------------------------------------

        subset = data[data[best] == value]

        # --------------------------------------------------
        # Supprimer feature utilisée
        # --------------------------------------------------

        remaining_features = [

            f for f in features

            if f != best
        ]

        # --------------------------------------------------
        # APPEL RÉCURSIF
        # --------------------------------------------------
        #
        # construire sous-arbre
        #
        # --------------------------------------------------

        subtree = decision_tree(

            subset,
            remaining_features,
            target
        )

        # --------------------------------------------------
        # Ajouter sous-arbre
        # --------------------------------------------------

        tree[best][value] = subtree

    # ------------------------------------------------------
    # Retourner arbre final
    # ------------------------------------------------------

    return tree

# ==========================================================
# 6. ENTRAINEMENT
# ==========================================================

# ----------------------------------------------------------
# Liste des features
# ----------------------------------------------------------

features = [

    'Meteo',
    'Temperature',
    'Humidite',
    'Vent'
]

# ----------------------------------------------------------
# Variable cible
# ----------------------------------------------------------

target = 'Jouer'

# ----------------------------------------------------------
# Construction arbre
# ----------------------------------------------------------

tree = decision_tree(
    df,
    features,
    target
)

# ==========================================================
# 7. AFFICHAGE
# ==========================================================

print("\n===== ARBRE DE DÉCISION =====")

print(tree)

# ==========================================================
# 8. PRÉDICTION
# ==========================================================

def predict(tree, sample):

    """
    ------------------------------------------------------
    Fonction de prédiction
    ------------------------------------------------------

    Elle parcourt l'arbre
    jusqu'à trouver
    une feuille finale.
    """

    # ------------------------------------------------------
    # Récupérer racine
    # ------------------------------------------------------

    root = list(tree.keys())[0]

    # ------------------------------------------------------
    # Valeur de la feature
    # dans l'exemple
    # ------------------------------------------------------

    value = sample[root]

    # ------------------------------------------------------
    # Aller dans branche correspondante
    # ------------------------------------------------------

    subtree = tree[root][value]

    # ------------------------------------------------------
    # Si feuille finale
    # ------------------------------------------------------

    if not isinstance(subtree, dict):

        return subtree

    # ------------------------------------------------------
    # Sinon continuer récursivement
    # ------------------------------------------------------

    return predict(subtree, sample)

# ==========================================================
# 9. TEST
# ==========================================================

# ----------------------------------------------------------
# Nouvel exemple
# ----------------------------------------------------------

sample = {

    'Meteo': 'Pluie',

    'Temperature': 'Froid',

    'Humidite': 'Normale',

    'Vent': 'Faible'
}

# ----------------------------------------------------------
# Faire prédiction
# ----------------------------------------------------------

prediction = predict(
    tree,
    sample
)

# ==========================================================
# 10. AFFICHAGE RÉSULTAT
# ==========================================================

print("\n===== TEST =====")

print("Exemple :", sample)

print("Classe prédite :", prediction)

===== DATASET =====
     Meteo Temperature Humidite    Vent Jouer
0   Soleil       Chaud    Haute  Faible   Non
1   Soleil       Chaud    Haute    Fort   Non
2  Nuageux       Chaud    Haute  Faible   Oui
3    Pluie       Moyen    Haute  Faible   Oui
4    Pluie       Froid  Normale  Faible   Oui
5    Pluie       Froid  Normale    Fort   Non
6  Nuageux       Froid  Normale    Fort   Oui
7   Soleil       Moyen    Haute  Faible   Non
8   Soleil       Froid  Normale  Faible   Oui
9    Pluie       Moyen  Normale  Faible   Oui

===== GAINS =====
Meteo : 0.3219
Temperature : 0.0955
Humidite : 0.1245
Vent : 0.0913

===== GAINS =====
Temperature : 0.8113
Humidite : 0.8113
Vent : 0.1226

===== GAINS =====
Temperature : 0.3113
Humidite : 0.1226
Vent : 0.8113

===== ARBRE DE DÉCISION =====
{'Meteo': {'Soleil': {'Temperature': {'Chaud': 'Non', 'Moyen': 'Non', 'Froid': 'Oui'}}, 'Nuageux': 'Oui', 'Pluie': {'Vent': {'Faible': 'Oui', 'Fort': 'Non'}}}}

===== TEST =====
Exemple : {'Meteo': 'Pluie', 'Temp